## 개념 · Seq2Seq — 문장을 벡터 하나로 줄였다가 풀기

- **인코더** — 입력 단어를 하나씩 읽어 문장 전체를 벡터 하나(context)로 압축
- **디코더** — 그 벡터와 직전 단어를 보고 출력 단어를 하나씩 생성 (`<sos>`로 시작, `<eos>`에서 끝)
- **문제** — 문장이 길수록 벡터 하나에 다 담지 못해 앞부분을 잊음 (병목) → 어텐션 등장

## 개념 · 어텐션 — 단어를 만들 때마다 입력을 다시 보기

- 단어를 하나 만들 때마다 인코더의 모든 단어 출력을 다시 봄
- "je" 만들 땐 "I", "étudiant" 만들 땐 "student"에 집중

1. 입력 단어마다 점수
2. softmax → 가중치 (합 = 1)
3. 가중치 × 인코더 출력의 합 = context

**그림과 코드의 차이**: 위 ①②③은 셀 2 BahdanauAttention 그대로. 원래 그림은 context를 LSTM 출력 뒤(Dense)에 쓰지만, 코드(셀 3)는 LSTM 입력 옆에 붙임

## [프로그램 5-11] 토이 번역 데이터 구성하기

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

pairs = [
    ("i love you", "나는 너를 사랑해"),
    ("he is happy", "그는 행복하다"),
    ("she is a student", "그녀는 학생이다"),
    ("thank you", "고마워"),
    ("good morning", "좋은 아침")
]

# 간단한 토크나이저
def tokenize(sentence):
    return sentence.split()

# 단어 사전 구성
src_vocab = {"<pad>":0, "<sos>":1, "<eos>":2}
tgt_vocab = {"<pad>":0, "<sos>":1, "<eos>":2}

for src, tgt in pairs:
    for w in tokenize(src):
        if w not in src_vocab:
            src_vocab[w] = len(src_vocab)
    for w in tokenize(tgt):
        if w not in tgt_vocab:
            tgt_vocab[w] = len(tgt_vocab)

inv_tgt = {v:k for k,v in tgt_vocab.items()}

max_len = 8

def encode(sentence, vocab):
    tokens = ["<sos>"] + tokenize(sentence) + ["<eos>"]
    tokens = tokens[:max_len] + ["<pad>"]*(max_len - len(tokens))
    return torch.tensor([vocab[t] for t in tokens])

class ToyTranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_encoded = encode(src, src_vocab)     # shape: (max_len,)
        tgt_encoded = encode(tgt, tgt_vocab)     # shape: (max_len,)
        return src_encoded, tgt_encoded

dataset = ToyTranslationDataset(pairs)
loader = DataLoader(dataset, batch_size=2, shuffle=True)



### 셀 1 설명 — 문장 쌍과 토크나이저

- 영어–한국어 문장 5쌍이 데이터의 전부 (연습용 초소형)
- `split()` — 띄어쓰기로 자르기: "i love you" → ['i', 'love', 'you']
- 한국어도 띄어쓰기 단위: "나는 너를 사랑해" → 단어 3개

**단어 사전 만들기**
- 특수 토큰 3개: `<pad>` 0 빈칸 · `<sos>` 1 시작 · `<eos>` 2 끝
- 처음 보는 단어에 `len(vocab)` = 다음 번호. 이미 있으면 건너뜀
- `inv_tgt` — 번호 → 단어 (번역 결과를 글자로 되돌릴 때)

- 영어 15개: 특수 0~2 · i 3 · love 4 · you 5 · he 6 · is 7 · happy 8 · she 9 · a 10 · student 11 · thank 12 · good 13 · morning 14
- 한국어 13개: 특수 0~2 · 나는 3 · 너를 4 · 사랑해 5 · 그는 6 · 행복하다 7 · 그녀는 8 · 학생이다 9 · 고마워 10 · 좋은 11 · 아침 12

**encode: 문장 → 번호 8개**

| 위치 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| 토큰 | `<sos>` | i | love | you | `<eos>` | `<pad>` | `<pad>` | `<pad>` |
| 번호 | 1 | 3 | 4 | 5 | 2 | 0 | 0 | 0 |

- 앞뒤에 `<sos>`, `<eos>` 붙이기
- 8개로 맞추기 — 길면 자르고, 짧으면 `<pad>`로 채우기
- 사전에서 번호를 찾아 텐서로 → (8,)

토큰이 8개를 넘는 문장(단어 7개 이상)은 잘리면서 `<eos>`가 사라집니다. 이 데이터는 가장 긴 문장도 토큰 6개라 괜찮습니다.

**Dataset과 DataLoader**

| | dataset[i] | loader 한 번 |
|---|---|---|
| src | (8,) | (2, 8) |
| tgt | (8,) | (2, 8) |

- Dataset 3종 세트 — U-Net 실습과 같은 구조
- 5쌍 ÷ 2 → 한 에폭에 배치 3개 (2쌍, 2쌍, 1쌍)

## [프로그램 5-12] Bahdanau 어텐션을 파이토치 모듈로 구현하기


In [2]:
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W1 = nn.Linear(hidden_size, hidden_size)
        self.W2 = nn.Linear(hidden_size, hidden_size)
        self.V  = nn.Linear(hidden_size, 1)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (B, H)
        # encoder_outputs: (B, T, H)
        # score: (B, T, 1)
        score = self.V(torch.tanh(
            self.W1(encoder_outputs) + self.W2(decoder_hidden).unsqueeze(1)
        ))
        attn_weights = F.softmax(score, dim=1)         # (B, T, 1)
        context = (attn_weights * encoder_outputs).sum(dim=1)   # (B, H)
        return context, attn_weights



### 셀 2 설명 — Bahdanau 어텐션

| 층 | 크기 | 하는 일 |
|---|---|---|
| W1 | H → H | 인코더 출력(단어 8개)을 변환 |
| W2 | H → H | 디코더의 지금 상태를 변환 |
| V | H → 1 | 합친 결과를 점수 1개로 |

- H = hidden_size = 64 (셀 4에서 정함)
- "어느 입력 단어가 지금 중요한가"를 판단하는 방법을 이 세 층이 학습으로 배움
- 세 층 모두 4장에서 본 `nn.Linear`입니다. 여기서는 "점수 계산기"로 쓰입니다.

**단어마다 점수 1개**

| 계산 | 모양 |
|---|---|
| W1(encoder_outputs) | (B, 8, H) |
| W2(decoder_hidden) | (B, H) |
| .unsqueeze(1) | (B, 1, H) |
| 두 개 더하기 | (B, 8, H) |
| tanh → V | (B, 8, 1) |

- T = 8: 입력 단어 8칸마다 점수 1개
- (B, 1, H)를 더하면 8칸 모두에 자동 복사되어 더해짐 (브로드캐스트)

**softmax와 context**
- `softmax(dim=1)` — 단어 8칸 방향으로 합이 1인 가중치
- (가중치 × 인코더 출력).sum(dim=1) → context (B, H) = 지금 필요한 입력 요약

실제로 찍어 본 "너를" 가중치 (100 에폭, 1회):

| `<sos>` | i | love | you | `<eos>` | `<pad>` | `<pad>` | `<pad>` |
|---|---|---|---|---|---|---|---|
| 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.02 | 0.21 | 0.76 |

기대와 달리 뒤쪽 `<pad>`에 몰림. 이상적이라면 you 칸이 커야 하지만, 5쌍을 외우기만 하면 돼서 단어 짝을 맞출 필요가 없습니다. 한 방향 LSTM의 뒤쪽 칸은 문장 전체를 읽은 요약이라 거기를 보는 게 가장 쉽습니다 (`<pad>`를 가려도 대개 `<eos>`로 몰림). 직접 확인 → 과제 3

## [프로그램 5-13] 인코더-디코더 모델 구성하기


In [3]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm  = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        # x: (B, T)
        x = self.embed(x)                 # (B, T, E)
        outputs, (h, c) = self.lstm(x)    # outputs: (B, T, H)
        return outputs, (h, c)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, attention):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm  = nn.LSTM(embed_size + hidden_size, hidden_size, batch_first=True)
        self.attn  = attention
        self.fc    = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        # x: (B,) 단일 토큰
        x = self.embed(x).unsqueeze(1)      # (B,1,E)

        # 디코더 현재 은닉 상태
        decoder_hidden = hidden[-1]         # (B,H)

        # 어텐션 계산
        context, _ = self.attn(decoder_hidden, encoder_outputs)  # (B,H)
        context = context.unsqueeze(1)       # (B,1,H)

        # LSTM 입력 결합
        lstm_input = torch.cat([x, context], dim=2)  # (B,1,E+H)

        outputs, (h, c) = self.lstm(lstm_input, (hidden, cell))  # outputs: (B,1,H)
        logits = self.fc(outputs.squeeze(1))                     # (B,V)
        return logits, (h, c)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, (h, c) = self.encoder(src)

        batch_size = src.size(0)
        T = tgt.size(1)

        outputs = []
        token = tgt[:, 0]     # <sos>

        for t in range(1, T):
            logits, (h, c) = self.decoder(token, h, c, encoder_outputs)
            outputs.append(logits.unsqueeze(1))
            token = tgt[:, t]     # teacher forcing

        return torch.cat(outputs, dim=1)   # (B, T-1, V)



### 셀 3 설명 — Encoder

흐름: 입력 x(B, 8) → embed(B, 8, 32) → lstm outputs(B, 8, 64)

- `Embedding` — 단어 번호 → 길이 E(=32) 벡터
- `outputs` — 모든 단어의 출력 → 어텐션이 매번 참고
- `(h, c)` — 마지막 상태 (1, B, 64) → 디코더의 시작 상태

**Decoder — 부품 준비**
- LSTM 입력 = E + H = 32 + 64 = 96 — 단어 벡터 옆에 context를 붙여 넣기 때문
- `fc = Linear(H, V)` — 64 → 13: 한국어 사전의 단어 13개 각각의 점수
- `attn` — 밖에서 만든 어텐션을 받아 보관

숫자 크기만 따라가 보기: 단어+context(32+64) → LSTM 입력(96) → LSTM 출력(64) → fc 출력(13)

**Decoder forward ① — 입력 준비와 어텐션**
- x — 단어 1개씩 (B,) → embed → unsqueeze(1) → (B, 1, E)
- `hidden[-1]` — (1, B, H)에서 마지막 층 → (B, H)
- 어텐션 → context (B, H) → unsqueeze(1) → (B, 1, H)

LSTM은 (배치, 시간, 특징) 3차원을 받습니다. 한 단어씩이라 시간 길이 1로 맞춥니다.

**Decoder forward ② — 한 칸 진행하고 점수 내기**

흐름: cat(B,1,96) → lstm(B,1,64) → squeeze(1)(B,64) → fc(B,13)

- `cat(..., dim=2)` — 특징 방향으로 붙이기: 32 + 64 = 96
- (hidden, cell)을 넣고 새 (h, c)를 받음 → 다음 단어 때 다시 넣음
- `logits (B, 13)` — 다음 단어 후보(사전 13개) 각각의 점수
- `squeeze(1)` — 길이 1인 시간 차원을 빼서 (B, 13)으로 반환. 이후 다시 unsqueeze(1) 해 모으고, 마지막에 이어 붙임

**Seq2Seq — 한 단어씩 반복**
- 인코더는 1번만 실행 · 첫 입력은 `<sos>`
- t = 1~7, 디코더를 한 칸씩 → 점수 모으기
- 다음 입력 = 예측이 아닌 정답 단어 (teacher forcing)
- 7개를 이어 붙여 (B, 7, 13)

**teacher forcing은 학습 때만**: 번역(셀 6)에선 정답이 없어 모델이 방금 고른 단어를 다음 입력으로 넣음

## [프로그램 5-14] 학습 루프 실행을 위한 학습 준비하기


In [4]:
embed_size = 32
hidden_size = 64

attention = BahdanauAttention(hidden_size)
encoder    = Encoder(len(src_vocab), embed_size, hidden_size)
decoder    = Decoder(len(tgt_vocab), embed_size, hidden_size, attention)
model      = Seq2Seq(encoder, decoder)

criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab["<pad>"])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



### 셀 4 설명 — 부품 조립과 손실 함수

- `embed_size = 32`, `hidden_size = 64` — 단어 벡터 32칸, LSTM 기억 64칸
- `len(src_vocab) = 15`, `len(tgt_vocab) = 13` — 사전 크기
- `ignore_index=0` (`tgt_vocab["<pad>"]` = 0) — 정답이 `<pad>`인 자리는 손실에서 제외 (빈칸 맞히기로 점수 따기 방지)

## [프로그램 5-15] 학습 루프 실행하기

In [5]:
num_epochs = 30

for epoch in range(num_epochs):
    total_loss = 0.0
    for src, tgt in loader:
        optimizer.zero_grad()
        logits = model(src, tgt)                # (B, T-1, V)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt[:, 1:].reshape(-1)              # <sos> 다음 토큰부터 비교
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")



Epoch 1: Loss = 7.8034
Epoch 2: Loss = 7.6091
Epoch 3: Loss = 7.4532
Epoch 4: Loss = 7.3215
Epoch 5: Loss = 7.1469
Epoch 6: Loss = 6.9426
Epoch 7: Loss = 6.7326
Epoch 8: Loss = 6.5886
Epoch 9: Loss = 6.3228
Epoch 10: Loss = 5.9983
Epoch 11: Loss = 5.4999
Epoch 12: Loss = 5.6573
Epoch 13: Loss = 5.4624
Epoch 14: Loss = 5.2782
Epoch 15: Loss = 4.7003
Epoch 16: Loss = 4.7942
Epoch 17: Loss = 4.5801
Epoch 18: Loss = 4.4440
Epoch 19: Loss = 4.0724
Epoch 20: Loss = 3.9338
Epoch 21: Loss = 3.7174
Epoch 22: Loss = 3.5762
Epoch 23: Loss = 3.2496
Epoch 24: Loss = 3.0956
Epoch 25: Loss = 2.9620
Epoch 26: Loss = 2.7834
Epoch 27: Loss = 2.6067
Epoch 28: Loss = 2.3111
Epoch 29: Loss = 2.2898
Epoch 30: Loss = 2.1369


### 셀 5 설명 — 모양 맞추기 reshape

B = 2일 때:

| | 처음 | reshape |
|---|---|---|
| 예측 logits | (2, 7, 13) | (14, 13) |
| 정답 tgt[:, 1:] | (2, 7) | (14,) |

- `CrossEntropyLoss`는 (N, 클래스 수)와 (N,)을 원함 → 단어 14개를 각각 13지선다로 (정답이 `<pad>`인 자리는 ignore_index로 빠짐)
- `tgt[:, 1:]` — `<sos>`를 뺀 7개. 예측도 t=1~7이라 짝이 맞음
- `-1` — 나머지로 자동 계산 (2×7=14)
- 학습 루프는 4장·U-Net과 같은 5줄, 순서도 같음

**에폭 수가 모자라다** — 무작위 시작값(seed)을 바꿔 10번씩 돌려 본 결과:

| 에폭 | Loss 처음 → 끝 | 정답 번역 |
|---|---|---|
| 30 (원본) | 7.7 → 약 2 | 3 / 10 |
| 50 | 7.7 → 약 0.6 | 10 / 10 |
| 100 | 7.7 → 약 0.1 | 10 / 10 |

- 정답 번역 = "i love you" → "나는 너를 사랑해"가 나온 횟수
- 30 에폭으론 덜 배워서 자주 틀림 → num_epochs = 100으로 실행

Loss가 0 가까이 가도 5쌍을 외운 것뿐 — 새 문장은 번역하지 못합니다 (과적합).

## [프로그램 5-16] 문장 번역하기

In [6]:
def translate(sentence):
    src = encode(sentence, src_vocab).unsqueeze(0)
    encoder_outputs, (h, c) = encoder(src)

    token = torch.tensor([tgt_vocab["<sos>"]])
    result = []

    for _ in range(max_len):
        logits, (h, c) = decoder(token, h, c, encoder_outputs)
        token = logits.argmax(dim=1)
        word = inv_tgt[token.item()]
        if word == "<eos>":
            break
        result.append(word)
    return " ".join(result)

print(translate("i love you"))
print(translate("good morning"))



나는 너를 사랑해
고마워


### 셀 6 설명 — 번역하기

- `encode(...)` (8,) → `unsqueeze(0)` → (1, 8) 배치 1개
- 인코더 1번 실행 → `encoder_outputs` (1, 8, 64)와 (h, c)
- 첫 입력은 `<sos>` → `tensor([1])`
- `result` — 만든 단어를 모을 리스트

**model.eval() · torch.no_grad()가 없다**: U-Net과 달리 빠져 있지만 결과는 같음. 역전파용 기록만 쓸데없이 남음 → 과제 3에서 `.detach()`가 필요한 이유

- `argmax(dim=1)` — 점수가 가장 높은 단어 하나 선택
- 그 단어가 다음 입력 → 정답 대신 자기 예측
- `<eos>`가 나오면 멈춤 (최대 8번)
- `inv_tgt`로 번호 → 단어, join으로 문장

실행 결과 — 100 에폭 학습 시:
```
나는 너를 사랑해
좋은 아침
```
30 에폭 그대로면 "고마워"처럼 엉뚱한 답이 자주 나옵니다.

사전에 없는 단어 → KeyError. 예: `translate("i love him")`

## 과제 · 직접 바꿔 보기 — 실습 3

1. **("i am happy", "나는 행복하다")**: 셀 1 pairs에 추가하고 처음부터 실행 → 셀 6에서 "i am happy"와 학습에 없는 "he is a student"도 번역해 보기
2. **token = logits.argmax(dim=1)**: 셀 3의 57번 줄을 이렇게 바꿔 teacher forcing 끄기 → Loss 비교
3. **어텐션 가중치 확인**: 셀 6의 9번 줄 앞에 아래 두 줄을 8칸 들여써 넣기
   ```python
   _, w = decoder.attn(h[-1], encoder_outputs)
   print(w.squeeze().detach().numpy().round(2))
   ```
   → 8칸 중 어디? 왜 `<pad>`일까요?
4. **vocab.get(t, 0)**: 셀 1의 35번 줄 `vocab[t]` 대신 → 모르는 단어는 `<pad>`(0)로, KeyError 막기
5. **num_epochs = 30 → 100, 5**: 에폭 수에 따라 번역 결과가 어떻게 달라지나요?

- 셀 1~5를 바꿨으면 셀 4(모델 만들기)부터 다시 실행합니다 (셀 1이면 셀 1부터, 셀 2·3이면 그 셀부터).
- 셀 5만 다시 돌리면 이미 학습된 모델을 이어서 학습하게 되어 비교가 안 됩니다. 과제 3처럼 셀 6만 바꿨으면 셀 6만 다시 실행합니다.

## 참고 · 실습 중 에러가 나면

| 에러 · 증상 | 원인 | 해결 |
|---|---|---|
| KeyError: 'him' | 사전에 없는 단어로 번역 | 학습 문장의 단어만 / `vocab.get` |